# European Football Data Preprocessing
Merges 5 leagues (Premier League, La Liga, Bundesliga, Serie A, Ligue 1), 1993/94–2025/26, into analysis-ready tables for Tableau / dashboards.

Run cells top to bottom. Takes ~1 minute total.

## 1. Get the raw data (clone the public dataset repo)

In [ ]:
!rm -rf /content/football-datasets
!git clone --depth 1 https://github.com/datasets/football-datasets.git /content/football-datasets
!ls /content/football-datasets/datasets

## 2. Imports & paths (Colab-specific)

In [ ]:
import pandas as pd
import glob
import os

BASE = "/content/football-datasets/datasets"
OUT = "/content/processed"
os.makedirs(OUT, exist_ok=True)

LEAGUES = {
    "premier-league": "Premier League (England)",
    "la-liga": "La Liga (Spain)",
    "bundesliga": "Bundesliga (Germany)",
    "serie-a": "Serie A (Italy)",
    "ligue-1": "Ligue 1 (France)",
}

def season_code_to_label(code):
    """'9394' -> '1993/94', '0001' -> '2000/01', '2526' -> '2025/26'"""
    a, b = code[:2], code[2:]
    a_full = ("19" + a) if a in ("93","94","95","96","97","98","99") else ("20" + a)
    return f"{a_full}/{b}"

def season_code_to_startyear(code):
    a = code[:2]
    a_full = int(("19" + a) if a in ("93","94","95","96","97","98","99") else ("20" + a))
    return a_full

log_lines = []
def log(msg):
    print(msg)
    log_lines.append(msg)

## 3. Step 1 — Load & merge all season files across all 5 leagues

In [ ]:
frames = []
missing_files = []

for folder, league_name in LEAGUES.items():
    files = sorted(glob.glob(f"{BASE}/{folder}/season-*.csv"))
    for f in files:
        code = os.path.basename(f).replace("season-", "").replace(".csv", "")
        try:
            df = pd.read_csv(f)
        except Exception as e:
            missing_files.append((f, str(e)))
            continue
        df["League"] = league_name
        df["Season"] = season_code_to_label(code)
        df["SeasonStartYear"] = season_code_to_startyear(code)
        frames.append(df)

raw = pd.concat(frames, ignore_index=True, sort=False)
log(f"STEP 1 — Raw merge: {len(raw):,} matches loaded across {raw['League'].nunique()} leagues, "
    f"{raw['Season'].nunique()} seasons ({raw['SeasonStartYear'].min()}-{raw['SeasonStartYear'].max()+1}).")
if missing_files:
    log(f"  WARNING: {len(missing_files)} files failed to load: {missing_files}")

raw.head()

## 4. Step 2 — Clean & type-cast

In [ ]:
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce", dayfirst=False)
n_bad_dates = raw["Date"].isna().sum()
log(f"STEP 2 — Parsed Date column. Unparseable dates: {n_bad_dates}")

numeric_cols = ["FTHG","FTAG","HTHG","HTAG","HS","AS","HST","AST","HF","AF","HC","AC","HY","AY","HR","AR"]
for c in numeric_cols:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

before = len(raw)
raw = raw.dropna(subset=["FTHG","FTAG","HomeTeam","AwayTeam"])
after = len(raw)
log(f"STEP 2 — Dropped {before-after} rows missing core result/team fields. Remaining: {after:,}")

raw["HomeTeam"] = raw["HomeTeam"].astype(str).str.strip()
raw["AwayTeam"] = raw["AwayTeam"].astype(str).str.strip()

## 5. Step 3 — Feature engineering (match level)

In [ ]:
raw["TotalGoals"] = raw["FTHG"] + raw["FTAG"]
raw["GoalDiff_HomeMinusAway"] = raw["FTHG"] - raw["FTAG"]
raw["HomeWin"] = (raw["FTR"] == "H").astype(int)
raw["AwayWin"] = (raw["FTR"] == "A").astype(int)
raw["Draw"] = (raw["FTR"] == "D").astype(int)
raw["HasDetailedStats"] = raw["HS"].notna()

log("STEP 3 — Engineered match-level fields: TotalGoals, GoalDiff, HomeWin/AwayWin/Draw flags, HasDetailedStats.")
n_detailed = raw["HasDetailedStats"].sum()
log(f"  Matches with full shot/card statistics: {n_detailed:,} / {len(raw):,} "
    f"({n_detailed/len(raw)*100:.1f}%)")

detail_start = (raw[raw["HasDetailedStats"]]
                 .groupby("League")["SeasonStartYear"].min()
                 .to_dict())
log(f"  First season with detailed stats, per league: {detail_start}")

common_detail_start = max(detail_start.values())
log(f"  -> Common 'detailed era' across all 5 leagues starts in {common_detail_start}/{str(common_detail_start+1)[-2:]}.")

## 6. Step 4 — Save cleaned match-level dataset (full range, Dashboard 1 raw source)

In [ ]:
match_cols = ["Date","League","Season","SeasonStartYear","HomeTeam","AwayTeam",
              "FTHG","FTAG","FTR","HTHG","HTAG","HTR","TotalGoals","GoalDiff_HomeMinusAway",
              "HomeWin","AwayWin","Draw","HasDetailedStats",
              "Referee","HS","AS","HST","AST","HF","AF","HC","AC","HY","AY","HR","AR"]
match_level = raw[match_cols].sort_values(["League","Date"]).reset_index(drop=True)
match_level.to_csv(f"{OUT}/match_level_full.csv", index=False)
log(f"STEP 4 — Saved match_level_full.csv ({len(match_level):,} rows, {len(match_cols)} columns).")

## 7. Step 5 — Season x League summary (home advantage over time — Dashboard 1 source)

In [ ]:
season_summary = raw.groupby(["League","Season","SeasonStartYear"]).agg(
    MatchesPlayed=("FTR","count"),
    HomeWins=("HomeWin","sum"),
    AwayWins=("AwayWin","sum"),
    Draws=("Draw","sum"),
    AvgHomeGoals=("FTHG","mean"),
    AvgAwayGoals=("FTAG","mean"),
    AvgTotalGoals=("TotalGoals","mean"),
).reset_index()

season_summary["HomeWinPct"] = (season_summary["HomeWins"] / season_summary["MatchesPlayed"] * 100).round(2)
season_summary["AwayWinPct"] = (season_summary["AwayWins"] / season_summary["MatchesPlayed"] * 100).round(2)
season_summary["DrawPct"] = (season_summary["Draws"] / season_summary["MatchesPlayed"] * 100).round(2)
season_summary["HomeAdvantage_WinPctGap"] = (season_summary["HomeWinPct"] - season_summary["AwayWinPct"]).round(2)
season_summary["HomeAdvantage_GoalGap"] = (season_summary["AvgHomeGoals"] - season_summary["AvgAwayGoals"]).round(3)

season_summary = season_summary.sort_values(["League","SeasonStartYear"]).reset_index(drop=True)
season_summary.to_csv(f"{OUT}/season_league_summary.csv", index=False)
log(f"STEP 5 — Saved season_league_summary.csv ({len(season_summary):,} rows = 5 leagues x ~33 seasons).")

covid_seasons = ["2019/20", "2020/21"]
log(f"  NOTE: Seasons {covid_seasons} include matches played without spectators (COVID-19).")

season_summary.head()

## 8. Step 6 — Detailed subset (shots/cards/fouls — Dashboard 2 source), common era only

In [ ]:
detailed = raw[raw["SeasonStartYear"] >= common_detail_start].copy()
detailed = detailed[detailed["HasDetailedStats"]]

detailed["HomeShotAccuracy"] = (detailed["HST"] / detailed["HS"]).replace([float("inf")], None)
detailed["AwayShotAccuracy"] = (detailed["AST"] / detailed["AS"]).replace([float("inf")], None)
detailed["HomeShotConversion"] = (detailed["FTHG"] / detailed["HS"]).replace([float("inf")], None)
detailed["AwayShotConversion"] = (detailed["FTAG"] / detailed["AS"]).replace([float("inf")], None)

detail_cols = ["Date","League","Season","SeasonStartYear","HomeTeam","AwayTeam","Referee",
               "FTHG","FTAG","FTR","HS","AS","HST","AST","HF","AF","HC","AC","HY","AY","HR","AR",
               "HomeShotAccuracy","AwayShotAccuracy","HomeShotConversion","AwayShotConversion"]
detailed = detailed[detail_cols].sort_values(["League","Date"]).reset_index(drop=True)
detailed.to_csv(f"{OUT}/match_level_detailed.csv", index=False)
log(f"STEP 6 — Saved match_level_detailed.csv ({len(detailed):,} rows) covering seasons "
    f"{common_detail_start}/{str(common_detail_start+1)[-2:]} onward, all 5 leagues.")

home_side = detailed.groupby(["League","Season","HomeTeam"]).agg(
    HomeMatches=("FTR","count"),
    HomeGoalsFor=("FTHG","mean"),
    HomeGoalsAgainst=("FTAG","mean"),
    HomeShots=("HS","mean"),
    HomeShotsOnTarget=("HST","mean"),
    HomeCards=("HY","mean"),
).reset_index().rename(columns={"HomeTeam":"Team"})

away_side = detailed.groupby(["League","Season","AwayTeam"]).agg(
    AwayMatches=("FTR","count"),
    AwayGoalsFor=("FTAG","mean"),
    AwayGoalsAgainst=("FTHG","mean"),
    AwayShots=("AS","mean"),
    AwayShotsOnTarget=("AST","mean"),
    AwayCards=("AY","mean"),
).reset_index().rename(columns={"AwayTeam":"Team"})

team_season = pd.merge(home_side, away_side, on=["League","Season","Team"], how="outer")
team_season.to_csv(f"{OUT}/team_season_summary.csv", index=False)
log(f"STEP 6b — Saved team_season_summary.csv ({len(team_season):,} team-season rows).")

## 9. Step 7 — Referee summary (Story source)

In [ ]:
ref = detailed.dropna(subset=["Referee"])
ref = ref[ref["Referee"].astype(str).str.strip() != ""]

ref["TotalYellows"] = ref["HY"] + ref["AY"]
ref["TotalReds"] = ref["HR"] + ref["AR"]
ref["TotalFouls"] = ref["HF"] + ref["AF"]

referee_summary = ref.groupby(["Referee","League"]).agg(
    MatchesOfficiated=("FTR","count"),
    AvgYellowsPerMatch=("TotalYellows","mean"),
    AvgRedsPerMatch=("TotalReds","mean"),
    AvgFoulsPerMatch=("TotalFouls","mean"),
).reset_index()

referee_summary_reliable = referee_summary[referee_summary["MatchesOfficiated"] >= 30].copy()
referee_summary_reliable["AvgYellowsPerMatch"] = referee_summary_reliable["AvgYellowsPerMatch"].round(2)
referee_summary_reliable["AvgRedsPerMatch"] = referee_summary_reliable["AvgRedsPerMatch"].round(3)
referee_summary_reliable["AvgFoulsPerMatch"] = referee_summary_reliable["AvgFoulsPerMatch"].round(2)
referee_summary_reliable = referee_summary_reliable.sort_values("AvgYellowsPerMatch", ascending=False)
referee_summary_reliable.to_csv(f"{OUT}/referee_summary.csv", index=False)
log(f"STEP 7 — Saved referee_summary.csv ({len(referee_summary_reliable):,} referees with >=30 matches).")

## 10. Step 8 — Data quality report

In [ ]:
dq = []
dq.append(f"Total matches (all leagues, all seasons): {len(raw):,}")
dq.append(f"Date range: {raw['Date'].min().date()} to {raw['Date'].max().date()}")
dq.append(f"Leagues: {', '.join(LEAGUES.values())}")
dq.append(f"Seasons per league: {raw.groupby('League')['Season'].nunique().to_dict()}")
dq.append(f"Missing Referee field: {raw['Referee'].isna().sum():,} rows ({raw['Referee'].isna().mean()*100:.1f}%)")
dq.append(f"Missing shot stats (HS): {raw['HS'].isna().sum():,} rows ({raw['HS'].isna().mean()*100:.1f}%)")
dq.append(f"Detailed-era common start (all 5 leagues): {common_detail_start}/{str(common_detail_start+1)[-2:]}")

with open(f"{OUT}/PROCESSING_LOG.txt", "w") as f:
    f.write("DATA PREPROCESSING LOG\n")
    f.write("="*70 + "\n\n")
    f.write("\n".join(log_lines))
    f.write("\n\n" + "="*70 + "\n")
    f.write("DATA QUALITY SUMMARY\n")
    f.write("="*70 + "\n")
    f.write("\n".join(dq))

log("STEP 8 — Wrote PROCESSING_LOG.txt")
print("\n".join(dq))

## 11. Quick sanity check — does the home-advantage decline actually show up?

In [ ]:
s = season_summary.copy()
s['Era'] = pd.cut(s['SeasonStartYear'], bins=[1992,1999,2004,2009,2014,2019,2026],
                   labels=['1993-99','2000-04','2005-09','2010-14','2015-19','2020-26'])
print("Home advantage gap by era (all leagues avg):")
print(s.groupby('Era')['HomeAdvantage_WinPctGap'].mean().round(2))

print("\nCOVID seasons vs neighbors:")
covid = s[s['Season'].isin(['2018/19','2019/20','2020/21','2021/22'])]
print(covid.groupby('Season')['HomeAdvantage_WinPctGap'].mean().round(2))

## 11b. Per-league COVID home-advantage drop (key evidence for sub-question 3)
Compares each league's home-advantage gap before COVID (2017/18-2018/19 average) against
the fully-behind-closed-doors season (2020/21), and saves it as its own CSV so it can be
pulled straight into a dedicated Tableau worksheet (the league-comparison bar chart).

In [ ]:
pre_covid = (s[s['Season'].isin(['2017/18','2018/19'])]
             .groupby('League')['HomeAdvantage_WinPctGap'].mean()
             .rename('PreCovid_Avg_Gap'))
covid_2021 = (s[s['Season'] == '2020/21']
              .groupby('League')['HomeAdvantage_WinPctGap'].mean()
              .rename('Covid_2020_21_Gap'))

covid_drop = pd.concat([pre_covid, covid_2021], axis=1).reset_index()
covid_drop['Drop_Points'] = (covid_drop['PreCovid_Avg_Gap'] - covid_drop['Covid_2020_21_Gap']).round(2)
covid_drop['Drop_Pct'] = (covid_drop['Drop_Points'] / covid_drop['PreCovid_Avg_Gap'] * 100).round(1)
covid_drop['PreCovid_Avg_Gap'] = covid_drop['PreCovid_Avg_Gap'].round(2)
covid_drop['Covid_2020_21_Gap'] = covid_drop['Covid_2020_21_Gap'].round(2)
covid_drop = covid_drop.sort_values('Drop_Points', ascending=False).reset_index(drop=True)

covid_drop.to_csv(f"{OUT}/covid_league_comparison.csv", index=False)
step9_msg = (f"STEP 9 — Saved covid_league_comparison.csv ({len(covid_drop)} leagues). "
             f"Compares pre-COVID (2017/18-2018/19 avg) vs fully-closed-doors 2020/21 home advantage gap.")
log(step9_msg)

# PROCESSING_LOG.txt was already written in section 10 (Step 8), before this step existed --
# append this step's line so the log file stays complete and in sync.
with open(f"{OUT}/PROCESSING_LOG.txt", "a") as f:
    f.write("\n" + step9_msg)

covid_drop

## 11c. VAR technology effect — cards-per-foul before vs after introduction (Story evidence)
Tests whether VAR's rollout (real, documented introduction seasons per league) changed how
strictly fouls get punished. Uses cards issued *per foul committed*, not raw card counts,
since foul volume itself can shift independently and would muddy a raw-count comparison.

In [ ]:
var_start_year = {
    "Bundesliga (Germany)": 2017,
    "Serie A (Italy)": 2017,
    "La Liga (Spain)": 2018,
    "Ligue 1 (France)": 2018,
    "Premier League (England)": 2019,
}

var_df = detailed.copy()
var_df['TotalCards'] = var_df['HY'] + var_df['AY'] + var_df['HR'] + var_df['AR']
var_df['TotalFouls'] = var_df['HF'] + var_df['AF']
var_df['CardsPerFoul'] = var_df['TotalCards'] / var_df['TotalFouls']
var_df['VARIntroYear'] = var_df['League'].map(var_start_year)
var_df['Period'] = var_df['SeasonStartYear'].ge(var_df['VARIntroYear']).map({True: 'Post-VAR', False: 'Pre-VAR'})

var_summary = var_df.groupby(['League','Period']).agg(
    Matches=('CardsPerFoul','count'),
    AvgCardsPerFoul=('CardsPerFoul','mean'),
    AvgCardsPerMatch=('TotalCards','mean'),
    AvgFoulsPerMatch=('TotalFouls','mean'),
).reset_index()
var_summary['AvgCardsPerFoul'] = var_summary['AvgCardsPerFoul'].round(4)
var_summary['AvgCardsPerMatch'] = var_summary['AvgCardsPerMatch'].round(3)
var_summary['AvgFoulsPerMatch'] = var_summary['AvgFoulsPerMatch'].round(3)

# add each league's VAR introduction season as its own column, for chart annotations
var_summary['VARIntroYear'] = var_summary['League'].map(var_start_year)

var_summary.to_csv(f"{OUT}/var_technology_effect.csv", index=False)
step10_msg = (f"STEP 10 — Saved var_technology_effect.csv ({len(var_summary)} rows = 5 leagues x Pre/Post-VAR). "
              f"VAR introduction years used: {var_start_year}.")
log(step10_msg)
with open(f"{OUT}/PROCESSING_LOG.txt", "a") as f:
    f.write("\n" + step10_msg)

var_summary

## 11d. Attacking evolution summary — draws vs goals, by league and era (Dashboard A source)
This is now the headline dataset: are draws declining and goals rising, per league, over 30 years?
Aggregated to era buckets so the trend is readable at a glance, but built from the full
season-by-season table so Tableau can still show the finer-grained trend line if needed.

In [ ]:
s2 = season_summary.copy()
s2['Era'] = pd.cut(s2['SeasonStartYear'], bins=[1992,1999,2004,2009,2014,2019,2026],
                    labels=['1993-99','2000-04','2005-09','2010-14','2015-19','2020-26'])

attacking_evolution = s2.groupby(['League','Era'], observed=True).agg(
    AvgDrawPct=('DrawPct','mean'),
    AvgTotalGoals=('AvgTotalGoals','mean'),
    Seasons=('Season','count'),
).reset_index()
attacking_evolution['AvgDrawPct'] = attacking_evolution['AvgDrawPct'].round(2)
attacking_evolution['AvgTotalGoals'] = attacking_evolution['AvgTotalGoals'].round(3)

attacking_evolution.to_csv(f"{OUT}/attacking_evolution_summary.csv", index=False)
step11_msg = (f"STEP 11 — Saved attacking_evolution_summary.csv ({len(attacking_evolution)} rows = "
              f"5 leagues x 6 eras). Core dashboard-A metrics: AvgDrawPct (declining) vs "
              f"AvgTotalGoals (rising) per league per era.")
log(step11_msg)
with open(f"{OUT}/PROCESSING_LOG.txt", "a") as f:
    f.write("\n" + step11_msg)

attacking_evolution.head(10)

## 11e. VAR home-bias footnote check (secondary/confirmatory only — see notes below)
**Important framing note:** this specific metric (away-team card gap) is the same construction
another group in the course used to test their crowd/COVID hypothesis (they called it "referee
booking bias"). We are re-using the same metric definition here deliberately, applied to a
*different* intervention (VAR introduction, not COVID crowd removal), as an explicitly disclosed
confirmatory cross-check — NOT as a headline finding. Keep it framed as a footnote in the report
and Story, and state the source of the metric idea plainly.

In [ ]:
var_bias_df = detailed.copy()
var_bias_df['VARIntroYear'] = var_bias_df['League'].map(var_start_year)
var_bias_df['Period'] = var_bias_df['SeasonStartYear'].ge(var_bias_df['VARIntroYear']).map(
    {True: 'Post-VAR', False: 'Pre-VAR'})
var_bias_df['AwayCardGap'] = (var_bias_df['AY'] + var_bias_df['AR']) - (var_bias_df['HY'] + var_bias_df['HR'])

var_home_bias = var_bias_df.groupby(['League','Period'])['AwayCardGap'].mean().round(3).reset_index()
var_home_bias = var_home_bias.rename(columns={'AwayCardGap': 'AvgAwayCardGap'})

var_home_bias.to_csv(f"{OUT}/var_home_bias_footnote.csv", index=False)
step12_msg = (f"STEP 12 — Saved var_home_bias_footnote.csv ({len(var_home_bias)} rows). "
              f"SECONDARY/CONFIRMATORY ONLY — metric definition borrowed from another group's "
              f"COVID analysis, re-applied here to VAR introduction; disclose this explicitly "
              f"in the report, do not present as an independent headline finding.")
log(step12_msg)
with open(f"{OUT}/PROCESSING_LOG.txt", "a") as f:
    f.write("\n" + step12_msg)

var_home_bias

## 12. Download the output files to your computer

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('/content/processed_data', 'zip', OUT)
files.download('/content/processed_data.zip')

### (Optional) Save straight to your Google Drive instead
Uncomment and run this instead of/in addition to the download cell above.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree(OUT, '/content/drive/MyDrive/football_project_processed', dirs_exist_ok=True)
# print('Saved to Google Drive: MyDrive/football_project_processed')